# 02_train_baseline_model (NACC) — Updated

Trains a simple baseline 4-class classifier using `output/cohort.csv` produced by `01_build_cohort`.


Vascular Composite Engineering (E8)
Raw vascular risk variables were replaced with:
•	VASC_RISK_COUNT
•	VASC_MED_COUNT
•	STROKE_TIA_ANY
To reduce multicollinearity and improve balanced accuracy slightly (~+0.002–0.003).
Conclusion:
Aggregated vascular burden provided modest improvement but did not dramatically improve subtype separability.

In [32]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    f1_score,
    precision_recall_fscore_support
)

CLASS_NAMES = ["AD", "NoDementia_Hy", "PD_LBD", "Vascular"]
CLASS_TO_INT = {c: i for i, c in enumerate(CLASS_NAMES)}
INT_TO_CLASS = {i: c for c, i in CLASS_TO_INT.items()}


def ensure_numpy(y):
    if isinstance(y, (pd.Series, pd.DataFrame)):
        y = y.values
    return np.asarray(y)


def y_is_string_labels(y):
    y = ensure_numpy(y)
    return y.dtype.kind in {"U", "S", "O"} and isinstance(y[0], str)


def encode_labels_if_needed(y):
    y = ensure_numpy(y)
    if len(y) == 0:
        return y
    if y_is_string_labels(y):
        return np.array([CLASS_TO_INT[v] for v in y])
    return y.astype(int)


def decode_labels_if_needed(y, return_strings=False):
    y = ensure_numpy(y).astype(int)
    if return_strings:
        return np.array([INT_TO_CLASS[v] for v in y], dtype=object)
    return y


def eval_predictions(y_true, y_pred, class_names=CLASS_NAMES, title="Model"):
    y_true = ensure_numpy(y_true)
    y_pred = ensure_numpy(y_pred)

    # convert both to ints for metric computation
    y_true_int = encode_labels_if_needed(y_true)
    y_pred_int = encode_labels_if_needed(y_pred)

    print(f"\n=== {title} ===")
    print(classification_report(y_true_int, y_pred_int, target_names=class_names, digits=4))
    print("Balanced accuracy:", balanced_accuracy_score(y_true_int, y_pred_int))
    print("Macro F1:", f1_score(y_true_int, y_pred_int, average="macro"))
    print("Weighted F1:", f1_score(y_true_int, y_pred_int, average="weighted"))
    print("Confusion matrix:")
    print(confusion_matrix(y_true_int, y_pred_int))

    p, r, f, s = precision_recall_fscore_support(
        y_true_int, y_pred_int, labels=np.arange(len(class_names)), zero_division=0
    )
    per_class = pd.DataFrame({
        "class": class_names,
        "precision": p,
        "recall": r,
        "f1": f,
        "support": s
    })
    return per_class


def predict_with_class_thresholds(proba, thresholds=None, class_names=CLASS_NAMES):
    pred = np.argmax(proba, axis=1)

    if thresholds is None or len(thresholds) == 0:
        return pred

    for i in range(proba.shape[0]):
        candidates = []
        for cls_name, thr in thresholds.items():
            j = CLASS_TO_INT[cls_name]
            if proba[i, j] >= thr:
                candidates.append((proba[i, j] / max(thr, 1e-8), j))
        if candidates:
            pred[i] = max(candidates)[1]

    return pred


def search_best_single_threshold(
    y_true, proba, target_class="Vascular",
    grid=np.linspace(0.05, 0.60, 56),
    base_pred=None
):
    """
    Works whether y_true is strings or ints.
    """
    y_true = encode_labels_if_needed(y_true)
    j = CLASS_TO_INT[target_class]

    best = None
    for thr in grid:
        pred = np.argmax(proba, axis=1) if base_pred is None else base_pred.copy()
        pred[proba[:, j] >= thr] = j

        p, r, f, s = precision_recall_fscore_support(
            y_true, pred, labels=[j], zero_division=0
        )

        row = {
            "threshold": float(thr),
            "precision": float(p[0]),
            "recall": float(r[0]),
            "f1": float(f[0]),
            "support": int(s[0]),
        }

        if best is None or row["f1"] > best["f1"]:
            best = row

    return best

In [1]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier


exp_name = "E8b_vascular_signal"
H = 5  # horizon years used in build_cohort notebook

OUT_DIR = os.path.join("..", "output")

cohort_file = f"cohort_{exp_name}_H{H}.csv"
DATA_PATH = os.path.join(OUT_DIR, cohort_file)

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Could not find cohort file: {DATA_PATH}\n"
        f"Make sure 01_build_cohort exported: {cohort_file}"
    )

df = pd.read_csv(DATA_PATH)

print("Loaded:", cohort_file, df.shape)
print(df["label"].value_counts(dropna=False))
df.head()

Loaded: cohort_E8b_vascular_signal_H5.csv (38341, 69)
label
Censored         23300
NoDementia_Hy    11083
AD                3013
MixedOrOther       504
PD_LBD             289
Vascular           152
Name: count, dtype: int64


,NACCID,baseline_date,label,max_follow_years,has_H_followup,has_dementia_H,NACCAGE,SEX,RACE,HISPANIC,...,FOCLSIGN,HACHIN,HACHIN_GE_4,HACHIN_GE_7,STROKE_TIA_ANY,VASC_MED_COUNT,GAIT_ANY,NACCMOM,NACCDAD,SIVDFIND
0,NACC000011,2006-04-17,Censored,3.296372,False,False,62.0,2.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0,0.0,0,NaN,NaN,NaN
1,NACC000034,2015-07-16,NoDementia_Hy,6.986995,True,False,79.0,2.0,1.0,0.0,...,NaN,NaN,NaN,NaN,0,2.0,0,1.0,1.0,0.0
2,NACC000067,2012-05-23,Censored,2.088980,False,False,60.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0,1.0,1,1.0,0.0,NaN
3,NACC000073,2016-05-20,Censored,0.000000,False,False,44.0,2.0,1.0,0.0,...,NaN,NaN,NaN,NaN,0,0.0,0,0.0,0.0,0.0
4,NACC000138,2023-07-13,Censored,1.043121,False,False,54.0,2.0,50.0,1.0,...,NaN,NaN,NaN,NaN,0,0.0,0,0.0,0.0,0.0


In [2]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score, f1_score
from sklearn.utils.class_weight import compute_sample_weight

import matplotlib.pyplot as plt

FIG_DIR = os.path.join(OUT_DIR, "figs_baseline")
os.makedirs(FIG_DIR, exist_ok=True)

RANDOM_STATE = 42

## Keep only the 4-class task

In [3]:
def fix_categorical_codes(X):
    X = X.copy()
    CAT_CODE_VARS = [c for c in ["SEX", "RACE", "HISPANIC", "NACCUDSD"] if c in X.columns]
    STD_MISS = {-4, 9}

    for c in CAT_CODE_VARS:
        s = pd.to_numeric(X[c], errors="coerce")
        s = s.mask(s.isin(list(STD_MISS)), np.nan)
        X[c] = s

    if "RACE" in X.columns:
        X["RACE"] = X["RACE"].mask(X["RACE"].isin([99]), np.nan)

    for c in CAT_CODE_VARS:
        s = pd.to_numeric(X[c], errors="coerce")
        X[c] = s.map(lambda v: np.nan if pd.isna(v) else str(int(v))).astype(object)

    if "EDUC" in X.columns:
        educ_num = pd.to_numeric(X["EDUC"], errors="coerce")
        #X["EDUC_BIN"] = np.where(educ_num <= 12, "<=12", ">12").astype(object)

    X = X.replace({pd.NA: np.nan})
    for c in X.columns:
        if X[c].dtype == "bool":
            X[c] = X[c].astype(int)
    return X

In [4]:
import numpy as np
import pandas as pd


df4 = df.copy()  

# Keeping only non-censored outcome classes for training
KEEP = {"AD", "NoDementia_Hy", "PD_LBD", "Vascular"}
df4 = df4[df4["label"].isin(KEEP)].copy()

print("4-class rows:", len(df4))
print(df4["label"].value_counts())

# Dropping IDs/labels
#y = df4["label"].astype(str)
# drop_cols = [c for c in ["NACCID", "baseline_date", "label"] if c in df4.columns]
#drop_cols = [c for c in [ "NACCID", "baseline_date", "label", "has_H_followup", "has_dementia_H", "max_follow_years", "abs_days_from_baseline"] if c in df4.columns]


def prepare_experiment_df(df, exp_name):
    df = df.copy()

    # columns that are always dropped
    drop_cols = [c for c in ["NACCID","baseline_date","label","has_H_followup","has_dementia_H","max_follow_years","abs_days_from_baseline", "NACCUDSD"]
                 if c in df.columns]

    # E8: engineered vascular burden; dropping raw components to avoid collinearity
    if exp_name == "E8_clean_vascular":
        raw_vasc = ["DIABETES","HYPERTEN","HYPERCHO","CVAFIB","CBSTROKE","CBTIA","CVCHF","CVANGINA","CVBYPASS","CVHVALVE"]
        drop_cols += [c for c in raw_vasc if c in df.columns]

        raw_meds = ["NACCLIPL","NACCAC","NACCBETA","NACCCCBS","NACCDIUR","NACCANGI","NACCVASD"]
        drop_cols += [c for c in raw_meds if c in df.columns]

        # drop_cols += [c for c in ["GAITDIS"] if c in df.columns]

    # E9: exec ratio; Dropping raw Trails
    if exp_name == "E9_clean_exec_ratio":
        drop_cols += [c for c in ["TRAILA","TRAILB","TRAILA_missing","TRAILB_missing"] if c in df.columns]

    # E10: building exec-memory axis (train-only z-scoring later), then Dropping raw
    # Here, what is dropped is only marked AFTER computing engineered features per split.

    X = df.drop(columns=[c for c in drop_cols if c in df.columns])
    y = df["label"].astype(str)

    return X, y


from sklearn.model_selection import train_test_split

X, y = prepare_experiment_df(df4, exp_name)

# Applying categorical cleanup before splitting
X = fix_categorical_codes(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train label counts:\n", y_train.value_counts())
print("Test label counts:\n", y_test.value_counts())



def z_from_train(train_s, s):
    mu = train_s.mean()
    sd = train_s.std(ddof=0)
    if pd.isna(sd) or sd == 0:
        return s*0.0
    return (s - mu) / sd

if exp_name == "E10_exec_mem_axis_raw":
    needed = ["NACCMMSE", "NACCMOCA", "TRAILB"]
    missing = [c for c in needed if c not in X_train.columns]
    if missing:
        raise ValueError(f"E10 missing required columns: {missing}")

    # Z-scores using train stats only
    X_train["Z_TRAILB"] = z_from_train(X_train["TRAILB"], X_train["TRAILB"])
    X_test["Z_TRAILB"]  = z_from_train(X_train["TRAILB"], X_test["TRAILB"])

    mem_cols = [c for c in ["NACCMMSE","NACCMOCA"] if c in X_train.columns]
    Zm_train = []
    Zm_test  = []
    for c in mem_cols:
        Zm_train.append(z_from_train(X_train[c], X_train[c]))
        Zm_test.append(z_from_train(X_train[c], X_test[c]))

    X_train["Z_MEM"] = np.nanmean(np.vstack([z.values for z in Zm_train]), axis=0)
    X_test["Z_MEM"]  = np.nanmean(np.vstack([z.values for z in Zm_test]), axis=0)

    # Executive minus memory axis
    X_train["EXEC_MINUS_MEM"] = X_train["Z_TRAILB"] - X_train["Z_MEM"]
    X_test["EXEC_MINUS_MEM"]  = X_test["Z_TRAILB"]  - X_test["Z_MEM"]

    # Dropping raw components to avoid collinearity
    drop_raw = [
        "TRAILA","TRAILB","NACCMMSE","NACCMOCA",
        "TRAILA_missing","TRAILB_missing","NACCMMSE_missing","NACCMOCA_missing",
        "Z_TRAILB","Z_MEM"
    ]
    X_train = X_train.drop(columns=[c for c in drop_raw if c in X_train.columns])
    X_test  = X_test.drop(columns=[c for c in drop_raw if c in X_test.columns])

    print("E10 engineered EXEC_MINUS_MEM added. Shapes now:", X_train.shape, X_test.shape)

labels = ["AD", "NoDementia_Hy", "PD_LBD", "Vascular"]

4-class rows: 14537
label
NoDementia_Hy    11083
AD                3013
PD_LBD             289
Vascular           152
Name: count, dtype: int64
Train: (11629, 62) Test: (2908, 62)
Train label counts:
 label
NoDementia_Hy    8866
AD               2410
PD_LBD            231
Vascular          122
Name: count, dtype: int64
Test label counts:
 label
NoDementia_Hy    2217
AD                603
PD_LBD             58
Vascular           30
Name: count, dtype: int64


In [5]:
csf_cols = ["CSFABETA", "CSFPTAU", "CSFTTAU"]
present_csf_cols = [c for c in csf_cols if c in df.columns]

if len(present_csf_cols) == len(csf_cols):
    df_csf = df[df[present_csf_cols].notna().all(axis=1)].copy()
    print("CSF subset size:", len(df_csf))
    print(df_csf["label"].value_counts())
else:
    print("CSF columns not present in this dataset:", 
          [c for c in csf_cols if c not in df.columns])

CSF columns not present in this dataset: ['CSFABETA', 'CSFPTAU', 'CSFTTAU']


## Split + preprocess + train

Train/test split

In [6]:
# X_train, X_test, y_train, y_test = train_test_split(
#     X_all, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
# )

Preprocessing pipelines (sparse for LR/boosting)

In [7]:
# cat_cols = [c for c in X_train.columns if X_train[c].dtype == "object" or str(X_train[c].dtype).startswith("string")]
# num_cols = [c for c in X_train.columns if c not in cat_cols]

import pandas as pd

num_cols = [c for c in X_train.columns if pd.api.types.is_numeric_dtype(X_train[c])]
cat_cols = [c for c in X_train.columns if c not in num_cols]


numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ],
    remainder="drop"
)

print("Num cols:", len(num_cols), "Cat cols:", len(cat_cols))
print(sorted([c for c in X_train.columns if "VASC" in c or "EXEC" in c or "TRAIL" in c or "MMSE" in c or "MOCA" in c]))
print(sorted([c for c in X_train.columns]))



Num cols: 59 Cat cols: 3
['NACCMMSE', 'NACCMMSE_missing', 'NACCMOCA', 'NACCMOCA_missing', 'TRAILA', 'TRAILA_missing', 'TRAILB', 'TRAILB_missing', 'VASC', 'VASC_MED_COUNT']
['ANIMALS', 'APNEA', 'BEREM', 'BEVHALL', 'BOSTON', 'BRADY', 'CBSTROKE', 'CBTIA', 'CDRGLOB', 'CDRSUM', 'CDRSUM_missing', 'COGFLUC', 'CVAFIB', 'CVANGINA', 'CVANGIO', 'CVBYPASS', 'CVCHF', 'CVD', 'CVDCOG', 'CVDIMAG', 'CVHVALVE', 'CVOTHR', 'DEP2YRS', 'DIABETES', 'EDUC', 'FOCLSIGN', 'FOCLSYM', 'GAITDIS', 'GAIT_ANY', 'HACHIN', 'HACHIN_GE_4', 'HACHIN_GE_7', 'HISPANIC', 'HYPERCHO', 'HYPERTEN', 'IMAGEWMH', 'IMAGMWMH', 'NACCAGE', 'NACCDAD', 'NACCMMSE', 'NACCMMSE_missing', 'NACCMOCA', 'NACCMOCA_missing', 'NACCMOM', 'PARKGAIT', 'PARKSIGN', 'RACE', 'RESTTRL', 'RESTTRR', 'RIGIDL', 'RIGIDR', 'SEX', 'SIVDFIND', 'SLOWINGL', 'SLOWINGR', 'STROKE_TIA_ANY', 'TRAILA', 'TRAILA_missing', 'TRAILB', 'TRAILB_missing', 'VASC', 'VASC_MED_COUNT']


Model A: Logistic Regression (final settings)

# Section 3C: Logistic Regression (balanced weights + vascular boost + threshold)

In [8]:
# from sklearn.linear_model import LogisticRegression
# from sklearn.pipeline import Pipeline
# from sklearn.base import clone
# from sklearn.utils.class_weight import compute_class_weight
# from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score
# import numpy as np

# # 1) Systematic class weights (balanced) + optional vascular boost
# classes = np.unique(y_train)

# base_w = compute_class_weight(
#     class_weight="balanced",
#     classes=classes,
#     y=y_train
# )
# class_weight_dict = dict(zip(classes, base_w))

# VAS_MULTIPLIER = 1.1   # try: 1.0, 1.2, 1.3, 1.5
# class_weight_dict["Vascular"] = class_weight_dict.get("Vascular", 1.0) * VAS_MULTIPLIER

# print("Final class weights:", class_weight_dict)

# # 2) Fit multinomial logistic regression
# clf_lr = Pipeline(steps=[
#     ("preprocess", clone(preprocess)),
#     ("model", LogisticRegression(
#         solver="lbfgs",          # supports multinomial automatically
#         max_iter=8000,
#         class_weight=class_weight_dict,
#         C=0.1
#     ))
# ])

# clf_lr.fit(X_train, y_train)

# # 3) Default prediction
# pred_lr = clf_lr.predict(X_test)

# # 4) OPTIONAL: vascular threshold override (often boosts vascular recall)
# USE_VASC_THRESHOLD = False
# T_VAS = 0.18  # try: 0.15–0.25

# if USE_VASC_THRESHOLD:
#     probs = clf_lr.predict_proba(X_test)
#     order = clf_lr.named_steps["model"].classes_
#     vas_idx = list(order).index("Vascular")
#     p_vas = probs[:, vas_idx]

#     pred_lr = pred_lr.copy()
#     pred_lr[p_vas >= T_VAS] = "Vascular"

# print("\n=== Logistic Regression (balanced weights + vascular boost + optional threshold) ===")
# print(classification_report(y_test, pred_lr, digits=4, labels=labels))
# print("Balanced accuracy:", balanced_accuracy_score(y_test, pred_lr))
# print(confusion_matrix(y_test, pred_lr, labels=labels))

# Section 3A: Logistic Regression (custom weights, C=0.1)

In [9]:
from sklearn.linear_model import LogisticRegression

custom_w = {
    "NoDementia_Hy": 1.0,
    "AD": 2.0,
    "PD_LBD": 6.0,
    "Vascular": 8.0
}

clf_lr = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(
        max_iter=8000,
        class_weight=custom_w,
        C=0.1
    ))
])

clf_lr.fit(X_train, y_train)
pred_lr = clf_lr.predict(X_test)

print("=== Logistic Regression (custom weights, C=0.1) ===")
print(classification_report(y_test, pred_lr, digits=4))
print("Balanced accuracy:", balanced_accuracy_score(y_test, pred_lr))
print(confusion_matrix(y_test, pred_lr, labels=labels))


=== Logistic Regression (custom weights, C=0.1) ===
               precision    recall  f1-score   support

           AD     0.6235    0.7114    0.6646       603
NoDementia_Hy     0.9250    0.8791    0.9015      2217
       PD_LBD     0.4512    0.6379    0.5286        58
     Vascular     0.3226    0.3333    0.3279        30

     accuracy                         0.8339      2908
    macro avg     0.5806    0.6405    0.6056      2908
 weighted avg     0.8468    0.8339    0.8390      2908

Balanced accuracy: 0.6404557690758564
[[ 429  142   20   12]
 [ 235 1949   25    8]
 [  14    6   37    1]
 [  10   10    0   10]]


# Section 3G: Tuned Logistic Regression

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score
import numpy as np

custom_w = {
    "NoDementia_Hy": 1.0,
    "AD": 2.0,
    "PD_LBD": 6.0,
    "Vascular": 8.0
}

# 1) Created train/validation split from TRAIN only
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

# 2) Fitted LR on inner-train

clf_lr = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(
        max_iter=8000,
        class_weight=custom_w,
        C=0.1
    ))
])

clf_lr.fit(X_tr, y_tr)

# 3) Tuned vascular threshold on validation set
proba_val = clf_lr.predict_proba(X_val)
class_order = clf_lr.named_steps["model"].classes_
vas_idx = list(class_order).index("Vascular")

thresholds = np.linspace(0.05, 0.50, 20)
results = []

for T in thresholds:
    preds = []
    for p in proba_val:
        if p[vas_idx] >= T:
            preds.append("Vascular")
        else:
            preds.append(class_order[np.argmax(p)])
    ba = balanced_accuracy_score(y_val, preds)

    # vascular recall
    yv = np.array(y_val)
    pv = np.array(preds)
    denom = np.sum(yv == "Vascular")
    vrec = (np.sum((yv == "Vascular") & (pv == "Vascular")) / denom) if denom else 0.0

    results.append((T, ba, vrec))

best_T, best_ba, best_vrec = max(results, key=lambda x: x[1])
print(f"Best T_VAS (by val BA): {best_T:.3f} | val BA={best_ba:.4f} | val VascRecall={best_vrec:.4f}")

# 4) Retrained LR on FULL TRAIN and evaluate once on TEST using tuned threshold
clf_lr.fit(X_train, y_train)

class_order = clf_lr.named_steps["model"].classes_
vas_idx = list(class_order).index("Vascular")

proba_test = clf_lr.predict_proba(X_test)
preds_test = []

for p in proba_test:
    if p[vas_idx] >= best_T:
        preds_test.append("Vascular")
    else:
        preds_test.append(class_order[np.argmax(p)])

print("\n=== Logistic Regression (Vascular threshold tuned) ===")
print(classification_report(y_test, preds_test, digits=4, labels=labels))
print("Balanced accuracy:", balanced_accuracy_score(y_test, preds_test))
print(confusion_matrix(y_test, preds_test, labels=labels))

Best T_VAS (by val BA): 0.074 | val BA=0.6517 | val VascRecall=0.6000

=== Logistic Regression (Vascular threshold tuned) ===
               precision    recall  f1-score   support

           AD     0.6358    0.5473    0.5882       603
NoDementia_Hy     0.9331    0.8489    0.8890      2217
       PD_LBD     0.4861    0.6034    0.5385        58
     Vascular     0.0700    0.7000    0.1273        30

     accuracy                         0.7799      2908
    macro avg     0.5313    0.6749    0.5357      2908
 weighted avg     0.8536    0.7799    0.8118      2908

Balanced accuracy: 0.6749017151190526
[[ 330  124   17  132]
 [ 173 1882   20  142]
 [  13    5   35    5]
 [   3    6    0   21]]


In [11]:
# from sklearn.linear_model import LogisticRegression
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import (
#     classification_report,
#     confusion_matrix,
#     balanced_accuracy_score,
#     f1_score,
#     precision_recall_fscore_support
# )
# from sklearn.pipeline import Pipeline
# import numpy as np
# import pandas as pd

# # --------------------------------------------------
# # 0) Labels / weights
# # --------------------------------------------------
# labels = ["AD", "NoDementia_Hy", "PD_LBD", "Vascular"]

# # Slightly reduced vascular weight to help precision/F1
# custom_w = {
#     "NoDementia_Hy": 1.0,
#     "AD": 2.0,
#     "PD_LBD": 5.0,
#     "Vascular": 6.0
# }

# # --------------------------------------------------
# # 1) Train / validation split from TRAIN only
# # --------------------------------------------------
# X_tr, X_val, y_tr, y_val = train_test_split(
#     X_train, y_train,
#     test_size=0.2,
#     random_state=42,
#     stratify=y_train
# )

# # --------------------------------------------------
# # 2) Candidate model settings
# #    One model change included: Elastic Net logistic regression
# # --------------------------------------------------
# model_settings = [
#     {
#         "name": "LR_L2_C0.1",
#         "penalty": "l2",
#         "solver": "lbfgs",
#         "C": 0.1,
#         "l1_ratio": None,
#     },
#     {
#         "name": "LR_L2_C0.05",
#         "penalty": "l2",
#         "solver": "lbfgs",
#         "C": 0.05,
#         "l1_ratio": None,
#     },
#     {
#         "name": "LR_ElasticNet_C0.1_l1r0.2",
#         "penalty": "elasticnet",
#         "solver": "saga",
#         "C": 0.1,
#         "l1_ratio": 0.2,
#     },
#     {
#         "name": "LR_ElasticNet_C0.05_l1r0.2",
#         "penalty": "elasticnet",
#         "solver": "saga",
#         "C": 0.05,
#         "l1_ratio": 0.2,
#     },
# ]

# thresholds = np.linspace(0.05, 0.40, 36)

# # --------------------------------------------------
# # Helper: apply vascular threshold override
# # --------------------------------------------------
# def predict_with_vascular_threshold(proba, class_order, threshold):
#     vas_idx = list(class_order).index("Vascular")
#     preds = []
#     for p in proba:
#         if p[vas_idx] >= threshold:
#             preds.append("Vascular")
#         else:
#             preds.append(class_order[np.argmax(p)])
#     return np.array(preds)

# # --------------------------------------------------
# # 3) Validation search:
# #    choose model + threshold by vascular F1
# #    tie-breakers: macro F1, balanced accuracy
# # --------------------------------------------------
# search_rows = []

# for cfg in model_settings:
#     clf = Pipeline(steps=[
#         ("preprocess", preprocess),
#         ("model", LogisticRegression(
#             max_iter=12000,
#             class_weight=custom_w,
#             C=cfg["C"],
#             penalty=cfg["penalty"],
#             solver=cfg["solver"],
#             l1_ratio=cfg["l1_ratio"],
#             n_jobs=None
#         ))
#     ])

#     clf.fit(X_tr, y_tr)

#     proba_val = clf.predict_proba(X_val)
#     class_order = clf.named_steps["model"].classes_

#     for T in thresholds:
#         preds_val = predict_with_vascular_threshold(proba_val, class_order, T)

#         # overall metrics
#         ba = balanced_accuracy_score(y_val, preds_val)
#         macro_f1 = f1_score(y_val, preds_val, average="macro")

#         # per-class metrics
#         p_arr, r_arr, f_arr, s_arr = precision_recall_fscore_support(
#             y_val, preds_val, labels=labels, zero_division=0
#         )

#         metric_map = {
#             lab: {"precision": p, "recall": r, "f1": f, "support": s}
#             for lab, p, r, f, s in zip(labels, p_arr, r_arr, f_arr, s_arr)
#         }

#         vprec = metric_map["Vascular"]["precision"]
#         vrec  = metric_map["Vascular"]["recall"]
#         vf1   = metric_map["Vascular"]["f1"]

#         search_rows.append({
#             "model_name": cfg["name"],
#             "penalty": cfg["penalty"],
#             "solver": cfg["solver"],
#             "C": cfg["C"],
#             "l1_ratio": cfg["l1_ratio"],
#             "threshold": T,
#             "val_bal_acc": ba,
#             "val_macro_f1": macro_f1,
#             "val_vasc_precision": vprec,
#             "val_vasc_recall": vrec,
#             "val_vasc_f1": vf1,
#         })

# search_df = pd.DataFrame(search_rows)

# # Sort by vascular F1 first, then macro F1, then BA
# search_df = search_df.sort_values(
#     ["val_vasc_f1", "val_macro_f1", "val_bal_acc"],
#     ascending=[False, False, False]
# ).reset_index(drop=True)

# print("Top validation settings:")
# display(search_df.head(10))

# best_row = search_df.iloc[0]
# best_model_name = best_row["model_name"]
# best_T = float(best_row["threshold"])

# print(
#     f"Best setting: {best_model_name} | "
#     f"T_VAS={best_T:.3f} | "
#     f"val VascF1={best_row['val_vasc_f1']:.4f} | "
#     f"val MacroF1={best_row['val_macro_f1']:.4f} | "
#     f"val BA={best_row['val_bal_acc']:.4f} | "
#     f"val VascPrec={best_row['val_vasc_precision']:.4f} | "
#     f"val VascRec={best_row['val_vasc_recall']:.4f}"
# )

# # --------------------------------------------------
# # 4) Refit best model on FULL TRAIN
# # --------------------------------------------------
# best_cfg = next(cfg for cfg in model_settings if cfg["name"] == best_model_name)

# clf_best = Pipeline(steps=[
#     ("preprocess", preprocess),
#     ("model", LogisticRegression(
#         max_iter=12000,
#         class_weight=custom_w,
#         C=best_cfg["C"],
#         penalty=best_cfg["penalty"],
#         solver=best_cfg["solver"],
#         l1_ratio=best_cfg["l1_ratio"],
#         n_jobs=None
#     ))
# ])

# clf_best.fit(X_train, y_train)

# class_order = clf_best.named_steps["model"].classes_
# proba_test = clf_best.predict_proba(X_test)
# preds_test = predict_with_vascular_threshold(proba_test, class_order, best_T)

# # --------------------------------------------------
# # 5) Final test evaluation
# # --------------------------------------------------
# print("\n=== Logistic Regression (best model + vascular threshold tuned by Vascular F1) ===")
# print(classification_report(y_test, preds_test, digits=4, labels=labels, zero_division=0))

# test_bal_acc = balanced_accuracy_score(y_test, preds_test)
# test_macro_f1 = f1_score(y_test, preds_test, average="macro")
# test_weighted_f1 = f1_score(y_test, preds_test, average="weighted")
# test_cm = confusion_matrix(y_test, preds_test, labels=labels)

# print("Balanced accuracy:", test_bal_acc)
# print("Macro F1:", test_macro_f1)
# print("Weighted F1:", test_weighted_f1)
# print(confusion_matrix(y_test, preds_test, labels=labels))

# # Optional: print vascular-specific test metrics cleanly
# p_arr, r_arr, f_arr, s_arr = precision_recall_fscore_support(
#     y_test, preds_test, labels=labels, zero_division=0
# )
# test_metric_map = {
#     lab: {"precision": p, "recall": r, "f1": f, "support": s}
#     for lab, p, r, f, s in zip(labels, p_arr, r_arr, f_arr, s_arr)
# }

# print("\nTest vascular metrics:")
# print(test_metric_map["Vascular"])

In [12]:
# from sklearn.linear_model import LogisticRegression
# from sklearn.model_selection import StratifiedKFold
# from sklearn.metrics import (
#     classification_report,
#     confusion_matrix,
#     balanced_accuracy_score,
#     f1_score,
#     precision_recall_fscore_support
# )
# from sklearn.pipeline import Pipeline
# import numpy as np
# import pandas as pd

# # --------------------------------------------------
# # 0) Labels / weights
# # --------------------------------------------------
# labels = ["AD", "NoDementia_Hy", "PD_LBD", "Vascular"]

# custom_w = {
#     "NoDementia_Hy": 1.0,
#     "AD": 2.0,
#     "PD_LBD": 5.0,
#     "Vascular": 6.0
# }

# # --------------------------------------------------
# # 1) Candidate model settings
# # --------------------------------------------------
# model_settings = [
#     {
#         "name": "LR_L2_C0.1",
#         "penalty": "l2",
#         "solver": "lbfgs",
#         "C": 0.1,
#         "l1_ratio": None,
#     },
#     {
#         "name": "LR_L2_C0.05",
#         "penalty": "l2",
#         "solver": "lbfgs",
#         "C": 0.05,
#         "l1_ratio": None,
#     },
#     {
#         "name": "LR_ElasticNet_C0.1_l1r0.2",
#         "penalty": "elasticnet",
#         "solver": "saga",
#         "C": 0.1,
#         "l1_ratio": 0.2,
#     },
#     {
#         "name": "LR_ElasticNet_C0.05_l1r0.2",
#         "penalty": "elasticnet",
#         "solver": "saga",
#         "C": 0.05,
#         "l1_ratio": 0.2,
#     },
# ]

# thresholds = np.linspace(0.05, 0.40, 36)

# # --------------------------------------------------
# # Helper: apply vascular threshold override
# # --------------------------------------------------
# def predict_with_vascular_threshold(proba, class_order, threshold):
#     vas_idx = list(class_order).index("Vascular")
#     preds = []
#     for p in proba:
#         if p[vas_idx] >= threshold:
#             preds.append("Vascular")
#         else:
#             preds.append(class_order[np.argmax(p)])
#     return np.array(preds)

# # --------------------------------------------------
# # 2) 3-fold CV search (FAST VERSION)
# # --------------------------------------------------
# skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# cv_rows = []

# for cfg in model_settings:
#     print(f"Running model config: {cfg['name']}")
    
#     # Store metrics by threshold across folds
#     threshold_metric_store = {
#         T: {
#             "bal_acc": [],
#             "macro_f1": [],
#             "vasc_precision": [],
#             "vasc_recall": [],
#             "vasc_f1": []
#         }
#         for T in thresholds
#     }

#     for fold_id, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), start=1):
#         print(f"  Fold {fold_id}/3")

#         X_tr_fold = X_train.iloc[tr_idx]
#         X_val_fold = X_train.iloc[val_idx]
#         y_tr_fold = y_train.iloc[tr_idx]
#         y_val_fold = y_train.iloc[val_idx]

#         clf = Pipeline(steps=[
#             ("preprocess", preprocess),
#             ("model", LogisticRegression(
#                 max_iter=12000,
#                 class_weight=custom_w,
#                 C=cfg["C"],
#                 penalty=cfg["penalty"],
#                 solver=cfg["solver"],
#                 l1_ratio=cfg["l1_ratio"],
#                 n_jobs=None
#             ))
#         ])

#         # FIT ONCE PER FOLD
#         clf.fit(X_tr_fold, y_tr_fold)

#         # PREDICT PROBA ONCE PER FOLD
#         proba_val = clf.predict_proba(X_val_fold)
#         class_order = clf.named_steps["model"].classes_

#         # LOOP OVER THRESHOLDS USING SAME PROBAS
#         for T in thresholds:
#             preds_val = predict_with_vascular_threshold(proba_val, class_order, T)

#             ba = balanced_accuracy_score(y_val_fold, preds_val)
#             macro_f1 = f1_score(y_val_fold, preds_val, average="macro")

#             p_arr, r_arr, f_arr, s_arr = precision_recall_fscore_support(
#                 y_val_fold, preds_val, labels=labels, zero_division=0
#             )

#             metric_map = {
#                 lab: {"precision": p, "recall": r, "f1": f, "support": s}
#                 for lab, p, r, f, s in zip(labels, p_arr, r_arr, f_arr, s_arr)
#             }

#             threshold_metric_store[T]["bal_acc"].append(ba)
#             threshold_metric_store[T]["macro_f1"].append(macro_f1)
#             threshold_metric_store[T]["vasc_precision"].append(metric_map["Vascular"]["precision"])
#             threshold_metric_store[T]["vasc_recall"].append(metric_map["Vascular"]["recall"])
#             threshold_metric_store[T]["vasc_f1"].append(metric_map["Vascular"]["f1"])

#     # Aggregate across folds for this model config
#     for T in thresholds:
#         cv_rows.append({
#             "model_name": cfg["name"],
#             "penalty": cfg["penalty"],
#             "solver": cfg["solver"],
#             "C": cfg["C"],
#             "l1_ratio": cfg["l1_ratio"],
#             "threshold": T,
#             "cv_bal_acc_mean": np.mean(threshold_metric_store[T]["bal_acc"]),
#             "cv_bal_acc_std": np.std(threshold_metric_store[T]["bal_acc"]),
#             "cv_macro_f1_mean": np.mean(threshold_metric_store[T]["macro_f1"]),
#             "cv_macro_f1_std": np.std(threshold_metric_store[T]["macro_f1"]),
#             "cv_vasc_precision_mean": np.mean(threshold_metric_store[T]["vasc_precision"]),
#             "cv_vasc_recall_mean": np.mean(threshold_metric_store[T]["vasc_recall"]),
#             "cv_vasc_f1_mean": np.mean(threshold_metric_store[T]["vasc_f1"]),
#             "cv_vasc_f1_std": np.std(threshold_metric_store[T]["vasc_f1"]),
#         })

# cv_df = pd.DataFrame(cv_rows)

# # Sort by mean vascular F1 first, then mean macro F1, then mean BA
# cv_df = cv_df.sort_values(
#     ["cv_vasc_f1_mean", "cv_macro_f1_mean", "cv_bal_acc_mean"],
#     ascending=[False, False, False]
# ).reset_index(drop=True)

# print("\nTop 15 CV settings:")
# display(cv_df.head(15))

# best_row = cv_df.iloc[0]
# best_model_name = best_row["model_name"]
# best_T = float(best_row["threshold"])

# print(
#     f"Best CV setting: {best_model_name} | "
#     f"T_VAS={best_T:.3f} | "
#     f"CV mean VascF1={best_row['cv_vasc_f1_mean']:.4f} ± {best_row['cv_vasc_f1_std']:.4f} | "
#     f"CV mean MacroF1={best_row['cv_macro_f1_mean']:.4f} ± {best_row['cv_macro_f1_std']:.4f} | "
#     f"CV mean BA={best_row['cv_bal_acc_mean']:.4f} ± {best_row['cv_bal_acc_std']:.4f}"
# )

# # --------------------------------------------------
# # 3) Refit best model on FULL TRAIN
# # --------------------------------------------------
# best_cfg = next(cfg for cfg in model_settings if cfg["name"] == best_model_name)

# clf_best = Pipeline(steps=[
#     ("preprocess", preprocess),
#     ("model", LogisticRegression(
#         max_iter=12000,
#         class_weight=custom_w,
#         C=best_cfg["C"],
#         penalty=best_cfg["penalty"],
#         solver=best_cfg["solver"],
#         l1_ratio=best_cfg["l1_ratio"],
#         n_jobs=None
#     ))
# ])

# clf_best.fit(X_train, y_train)

# class_order = clf_best.named_steps["model"].classes_
# proba_test = clf_best.predict_proba(X_test)
# preds_test = predict_with_vascular_threshold(proba_test, class_order, best_T)

# # --------------------------------------------------
# # 4) Final test evaluation
# # --------------------------------------------------
# print("\n=== Logistic Regression (3-fold CV selected model + vascular threshold) ===")
# print(classification_report(y_test, preds_test, digits=4, labels=labels, zero_division=0))

# test_bal_acc = balanced_accuracy_score(y_test, preds_test)
# test_macro_f1 = f1_score(y_test, preds_test, average="macro")
# test_weighted_f1 = f1_score(y_test, preds_test, average="weighted")

# print("Balanced accuracy:", test_bal_acc)
# print("Macro F1:", test_macro_f1)
# print("Weighted F1:", test_weighted_f1)
# print(confusion_matrix(y_test, preds_test, labels=labels))

# p_arr, r_arr, f_arr, s_arr = precision_recall_fscore_support(
#     y_test, preds_test, labels=labels, zero_division=0
# )
# test_metric_map = {
#     lab: {"precision": p, "recall": r, "f1": f, "support": s}
#     for lab, p, r, f, s in zip(labels, p_arr, r_arr, f_arr, s_arr)
# }

# print("\nTest vascular metrics:")
# print(test_metric_map["Vascular"])

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    f1_score,
    precision_recall_fscore_support
)
from sklearn.pipeline import Pipeline
import numpy as np
import pandas as pd

# --------------------------------------------------
# 0) Labels / weights
# --------------------------------------------------
labels = ["AD", "NoDementia_Hy", "PD_LBD", "Vascular"]

# Slightly reduced vascular weight to help precision/F1
custom_w = {
    "NoDementia_Hy": 1.0,
    "AD": 2.0,
    "PD_LBD": 5.0,
    "Vascular": 6.0
}

# --------------------------------------------------
# 1) Train / validation split from TRAIN only
# --------------------------------------------------
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

# --------------------------------------------------
# 2) Candidate model settings
#    One model change included: Elastic Net logistic regression
# --------------------------------------------------
model_settings = [
    {
        "name": "LR_L2_C0.1",
        "penalty": "l2",
        "solver": "lbfgs",
        "C": 0.1,
        "l1_ratio": None,
    },
    {
        "name": "LR_L2_C0.05",
        "penalty": "l2",
        "solver": "lbfgs",
        "C": 0.05,
        "l1_ratio": None,
    },
    {
        "name": "LR_ElasticNet_C0.1_l1r0.2",
        "penalty": "elasticnet",
        "solver": "saga",
        "C": 0.1,
        "l1_ratio": 0.2,
    },
    {
        "name": "LR_ElasticNet_C0.05_l1r0.2",
        "penalty": "elasticnet",
        "solver": "saga",
        "C": 0.05,
        "l1_ratio": 0.2,
    },
]

thresholds = np.linspace(0.05, 0.40, 36)

# --------------------------------------------------
# Helper: apply vascular threshold override
# --------------------------------------------------
def predict_with_vascular_threshold(proba, class_order, threshold):
    vas_idx = list(class_order).index("Vascular")
    preds = []
    for p in proba:
        if p[vas_idx] >= threshold:
            preds.append("Vascular")
        else:
            preds.append(class_order[np.argmax(p)])
    return np.array(preds)

# --------------------------------------------------
# 3) Validation search:
#    choose model + threshold by vascular F1
#    tie-breakers: macro F1, balanced accuracy
# --------------------------------------------------
search_rows = []

for cfg in model_settings:
    clf = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", LogisticRegression(
            max_iter=12000,
            class_weight=custom_w,
            C=cfg["C"],
            penalty=cfg["penalty"],
            solver=cfg["solver"],
            l1_ratio=cfg["l1_ratio"],
            n_jobs=None
        ))
    ])

    clf.fit(X_tr, y_tr)

    proba_val = clf.predict_proba(X_val)
    class_order = clf.named_steps["model"].classes_

    for T in thresholds:
        preds_val = predict_with_vascular_threshold(proba_val, class_order, T)

        # overall metrics
        ba = balanced_accuracy_score(y_val, preds_val)
        macro_f1 = f1_score(y_val, preds_val, average="macro")

        # per-class metrics
        p_arr, r_arr, f_arr, s_arr = precision_recall_fscore_support(
            y_val, preds_val, labels=labels, zero_division=0
        )

        metric_map = {
            lab: {"precision": p, "recall": r, "f1": f, "support": s}
            for lab, p, r, f, s in zip(labels, p_arr, r_arr, f_arr, s_arr)
        }

        vprec = metric_map["Vascular"]["precision"]
        vrec  = metric_map["Vascular"]["recall"]
        vf1   = metric_map["Vascular"]["f1"]

        search_rows.append({
            "model_name": cfg["name"],
            "penalty": cfg["penalty"],
            "solver": cfg["solver"],
            "C": cfg["C"],
            "l1_ratio": cfg["l1_ratio"],
            "threshold": T,
            "val_bal_acc": ba,
            "val_macro_f1": macro_f1,
            "val_vasc_precision": vprec,
            "val_vasc_recall": vrec,
            "val_vasc_f1": vf1,
        })

search_df = pd.DataFrame(search_rows)

# Sort by vascular F1 first, then macro F1, then BA
search_df = search_df.sort_values(
    ["val_vasc_f1", "val_macro_f1", "val_bal_acc"],
    ascending=[False, False, False]
).reset_index(drop=True)

print("Top validation settings:")
display(search_df.head(10))

best_row = search_df.iloc[0]
best_model_name = best_row["model_name"]
best_T = float(best_row["threshold"])

print(
    f"Best setting: {best_model_name} | "
    f"T_VAS={best_T:.3f} | "
    f"val VascF1={best_row['val_vasc_f1']:.4f} | "
    f"val MacroF1={best_row['val_macro_f1']:.4f} | "
    f"val BA={best_row['val_bal_acc']:.4f} | "
    f"val VascPrec={best_row['val_vasc_precision']:.4f} | "
    f"val VascRec={best_row['val_vasc_recall']:.4f}"
)

# --------------------------------------------------
# 4) Refit best model on FULL TRAIN
# --------------------------------------------------
best_cfg = next(cfg for cfg in model_settings if cfg["name"] == best_model_name)

clf_best = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(
        max_iter=12000,
        class_weight=custom_w,
        C=best_cfg["C"],
        penalty=best_cfg["penalty"],
        solver=best_cfg["solver"],
        l1_ratio=best_cfg["l1_ratio"],
        n_jobs=None
    ))
])

clf_best.fit(X_train, y_train)

class_order = clf_best.named_steps["model"].classes_
proba_test = clf_best.predict_proba(X_test)
preds_test = predict_with_vascular_threshold(proba_test, class_order, best_T)

# --------------------------------------------------
# 5) Final test evaluation
# --------------------------------------------------
print("\n=== Logistic Regression (best model + vascular threshold tuned by Vascular F1) ===")
print(classification_report(y_test, preds_test, digits=4, labels=labels, zero_division=0))

test_bal_acc = balanced_accuracy_score(y_test, preds_test)
test_macro_f1 = f1_score(y_test, preds_test, average="macro")
test_weighted_f1 = f1_score(y_test, preds_test, average="weighted")
test_cm = confusion_matrix(y_test, preds_test, labels=labels)

print("Balanced accuracy:", test_bal_acc)
print("Macro F1:", test_macro_f1)
print("Weighted F1:", test_weighted_f1)
print(confusion_matrix(y_test, preds_test, labels=labels))

# Optional: print vascular-specific test metrics cleanly
p_arr, r_arr, f_arr, s_arr = precision_recall_fscore_support(
    y_test, preds_test, labels=labels, zero_division=0
)
test_metric_map = {
    lab: {"precision": p, "recall": r, "f1": f, "support": s}
    for lab, p, r, f, s in zip(labels, p_arr, r_arr, f_arr, s_arr)
}

print("\nTest vascular metrics:")
print(test_metric_map["Vascular"])

c:\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l2 with l1_ratio=None. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Python314\Lib\site-packag

Top validation settings:


,model_name,penalty,solver,C,l1_ratio,threshold,val_bal_acc,val_macro_f1,val_vasc_precision,val_vasc_recall,val_vasc_f1
0,LR_L2_C0.1,l2,lbfgs,0.10,NaN,0.26,0.635587,0.604633,0.191489,0.36,0.250000
1,LR_ElasticNet_C0.1_l1r0.2,elasticnet,saga,0.10,0.2,0.32,0.622413,0.604899,0.195122,0.32,0.242424
2,LR_L2_C0.05,l2,lbfgs,0.05,NaN,0.26,0.634786,0.601644,0.180000,0.36,0.240000
3,LR_ElasticNet_C0.1_l1r0.2,elasticnet,saga,0.10,0.2,0.27,0.621895,0.603664,0.190476,0.32,0.238806
4,LR_ElasticNet_C0.1_l1r0.2,elasticnet,saga,0.10,0.2,0.28,0.621895,0.603664,0.190476,0.32,0.238806
5,LR_ElasticNet_C0.1_l1r0.2,elasticnet,saga,0.10,0.2,0.29,0.621895,0.603664,0.190476,0.32,0.238806
6,LR_ElasticNet_C0.1_l1r0.2,elasticnet,saga,0.10,0.2,0.30,0.621895,0.603664,0.190476,0.32,0.238806
7,LR_ElasticNet_C0.1_l1r0.2,elasticnet,saga,0.10,0.2,0.31,0.621895,0.603664,0.190476,0.32,0.238806
8,LR_L2_C0.1,l2,lbfgs,0.10,NaN,0.25,0.634786,0.601025,0.176471,0.36,0.236842
9,LR_L2_C0.05,l2,lbfgs,0.05,NaN,0.25,0.634786,0.601025,0.176471,0.36,0.236842


Best setting: LR_L2_C0.1 | T_VAS=0.260 | val VascF1=0.2500 | val MacroF1=0.6046 | val BA=0.6356 | val VascPrec=0.1915 | val VascRec=0.3600


c:\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l2 with l1_ratio=None. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(



=== Logistic Regression (best model + vascular threshold tuned by Vascular F1) ===
               precision    recall  f1-score   support

           AD     0.6296    0.7048    0.6651       603
NoDementia_Hy     0.9255    0.8796    0.9019      2217
       PD_LBD     0.4737    0.6207    0.5373        58
     Vascular     0.2400    0.4000    0.3000        30

     accuracy                         0.8332      2908
    macro avg     0.5672    0.6513    0.6011      2908
 weighted avg     0.8481    0.8332    0.8393      2908

Balanced accuracy: 0.6512664811199783
Macro F1: 0.6010894499934994
Weighted F1: 0.8393491618452753
[[ 425  141   18   19]
 [ 228 1950   22   17]
 [  14    6   36    2]
 [   8   10    0   12]]

Test vascular metrics:
{'precision': np.float64(0.24), 'recall': np.float64(0.4), 'f1': np.float64(0.3), 'support': np.int64(30)}


In [37]:
def add_vascular_features(df):
    df = df.copy()

    # Use whichever names exist in your current dataset
    possible_cols = [
        "CBSTROKE", "CBTIA", "HYPERTEN", "DIABETES", "CVAFIB", "APNEA",
        "STROKE", "HYPERT", "DIABET", "AFIBRILL", "SLEEPAP"
    ]
    for c in possible_cols:
        if c in df.columns:
            df[c] = df[c].fillna(0)

    stroke_col = "CBSTROKE" if "CBSTROKE" in df.columns else ("STROKE" if "STROKE" in df.columns else None)
    tia_col    = "CBTIA" if "CBTIA" in df.columns else None
    htn_col    = "HYPERTEN" if "HYPERTEN" in df.columns else ("HYPERT" if "HYPERT" in df.columns else None)
    dm_col     = "DIABETES" if "DIABETES" in df.columns else ("DIABET" if "DIABET" in df.columns else None)
    af_col     = "CVAFIB" if "CVAFIB" in df.columns else ("AFIBRILL" if "AFIBRILL" in df.columns else None)
    ap_col     = "APNEA" if "APNEA" in df.columns else ("SLEEPAP" if "SLEEPAP" in df.columns else None)

    components = []
    for c in [stroke_col, tia_col, htn_col, dm_col, af_col, ap_col]:
        if c is not None:
            components.append(df[c].astype(float))

    if len(components) > 0:
        df["vascular_burden_score"] = np.sum(np.vstack(components), axis=0)

    if stroke_col is not None and htn_col is not None:
        df["stroke_x_htn"] = df[stroke_col].astype(float) * df[htn_col].astype(float)

    if dm_col is not None and htn_col is not None:
        df["dm_x_htn"] = df[dm_col].astype(float) * df[htn_col].astype(float)

    return df

X_train_fe = add_vascular_features(X_train)
X_val_fe   = add_vascular_features(X_val)
X_test_fe  = add_vascular_features(X_test)

In [38]:
def compute_vascular_risk_stratum(df):
    df = add_vascular_features(df)

    if "vascular_burden_score" not in df.columns:
        raise ValueError("vascular_burden_score was not created; check available columns.")

    # Example bins:
    # 0-1 -> low
    # 2-3 -> medium
    # 4+  -> high
    score = df["vascular_burden_score"].fillna(0)

    stratum = pd.Series(index=df.index, dtype="object")
    stratum[score <= 1] = "low"
    stratum[(score >= 2) & (score <= 3)] = "medium"
    stratum[score >= 4] = "high"
    return stratum

In [42]:
def fit_stratum_specific_thresholds(y_val, val_proba, val_df, target_class="Vascular"):
    j = CLASS_TO_INT[target_class]
    strata = compute_vascular_risk_stratum(val_df)

    # convert labels to ints once
    y_val_int = encode_labels_if_needed(y_val)

    thresholds = {}
    summary_rows = []

    for s in ["low", "medium", "high"]:
        mask = (strata == s).values
        y_sub = y_val_int[mask]
        proba_sub = val_proba[mask]

        if len(y_sub) == 0:
            continue

        # optional: skip strata with no vascular truth cases
        if np.sum(y_sub == j) == 0:
            continue

        best = None
        for thr in np.linspace(0.05, 0.70, 66):
            pred = np.argmax(proba_sub, axis=1)
            pred[proba_sub[:, j] >= thr] = j

            p, r, f, sup = precision_recall_fscore_support(
                y_sub, pred, labels=[j], zero_division=0
            )

            row = {
                "stratum": s,
                "threshold": float(thr),
                "precision": float(p[0]),
                "recall": float(r[0]),
                "f1": float(f[0]),
                "support": int(sup[0]),
            }

            if best is None or row["f1"] > best["f1"]:
                best = row

        if best is not None:
            thresholds[s] = best["threshold"]
            summary_rows.append(best)

    return thresholds, pd.DataFrame(summary_rows)

In [45]:
def predict_with_risk_stratified_margin_rule(proba, X_df, stratum_thresholds, margin=0.08):
    pred = np.argmax(proba, axis=1)
    vasc_idx = CLASS_TO_INT["Vascular"]
    strata = compute_vascular_risk_stratum(X_df)

    top2 = np.partition(proba, -2, axis=1)[:, -2:]
    top_prob = top2[:, 1]
    second_prob = top2[:, 0]

    for s in ["low", "medium", "high"]:
        if s not in stratum_thresholds:
            continue
        thr = stratum_thresholds[s]
        mask = (strata == s).values

        force_vasc = (
            mask &
            (proba[:, vasc_idx] >= thr) &
            ((top_prob - proba[:, vasc_idx]) <= margin)
        )
        pred[force_vasc] = vasc_idx

    return pred


In [46]:
X_val_fe = add_vascular_features(X_val)
X_test_fe = add_vascular_features(X_test)

val_proba_lr = clf_best.predict_proba(X_val)
stratum_thresholds_lr, table_lr = fit_stratum_specific_thresholds(
    y_val, val_proba_lr, X_val_fe, target_class="Vascular"
)
print(stratum_thresholds_lr)
display(table_lr)

test_proba_lr = clf_best.predict_proba(X_test)
test_pred_lr_strat = predict_with_risk_stratified_margin_rule(
    test_proba_lr,
    X_test_fe,
    stratum_thresholds_lr,
    margin=0.08
)

eval_predictions(
    y_test,
    test_pred_lr_strat,
    title="Current logistic model + risk-stratified vascular rule"
)

{'low': 0.09999999999999999, 'medium': 0.42999999999999994, 'high': 0.4799999999999999}


,stratum,threshold,precision,recall,f1,support
0,low,0.10,0.117647,0.545455,0.193548,11
1,medium,0.43,0.307692,0.444444,0.363636,9
2,high,0.48,0.555556,1.000000,0.714286,5



=== Current logistic model + risk-stratified vascular rule ===
               precision    recall  f1-score   support

           AD     0.6261    0.7164    0.6682       603
NoDementia_Hy     0.9243    0.8818    0.9026      2217
       PD_LBD     0.4800    0.6207    0.5414        58
     Vascular     0.3214    0.3000    0.3103        30

     accuracy                         0.8363      2908
    macro avg     0.5880    0.6297    0.6056      2908
 weighted avg     0.8474    0.8363    0.8407      2908

Balanced accuracy: 0.6297324619959322
Macro F1: 0.60562426975427
Weighted F1: 0.8406717371530384
Confusion matrix:
[[ 432  143   18   10]
 [ 233 1955   21    8]
 [  14    7   36    1]
 [  11   10    0    9]]


,class,precision,recall,f1,support
0,AD,0.626087,0.716418,0.668213,603
1,NoDementia_Hy,0.924350,0.881822,0.902585,2217
2,PD_LBD,0.480000,0.620690,0.541353,58
3,Vascular,0.321429,0.300000,0.310345,30


1) CatBoost

In [19]:
# pip install catboost
from catboost import CatBoostClassifier

# If your X is a DataFrame, CatBoost can use column names directly.
# If you have categorical columns, list them here; else leave empty.
cat_cols = []   # example: ["SEX", "RACE", "FORMVER"]
num_cols = [c for c in X_train.columns if c not in cat_cols]

cat_model = CatBoostClassifier(
    loss_function="MultiClass",
    eval_metric="TotalF1",
    iterations=2000,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=5.0,
    random_seed=42,
    verbose=200,
    auto_class_weights=None,
    class_weights=[1.0, 1.0, 2.0, 4.0]  # AD, NoDem, PD_LBD, Vascular
)

cat_model.fit(
    X_train,
    y_train,
    cat_features=cat_cols if len(cat_cols) > 0 else None,
    eval_set=(X_val, y_val),
    use_best_model=True
)

val_proba_cat = cat_model.predict_proba(X_val)
best_vasc_thr_cat = search_best_single_threshold(y_val, val_proba_cat, target_class="Vascular")
print("Best CatBoost vascular threshold on VAL:", best_vasc_thr_cat)

test_proba_cat = cat_model.predict_proba(X_test)
test_pred_cat = predict_with_class_thresholds(
    test_proba_cat,
    thresholds={"Vascular": best_vasc_thr_cat["threshold"]}
)

cat_results = eval_predictions(y_test, test_pred_cat, title="CatBoost + tuned vascular threshold")
cat_results

0:	learn: 0.7621672	test: 0.7622905	best: 0.7622905 (0)	total: 24.8ms	remaining: 49.5s
200:	learn: 0.8344558	test: 0.8331954	best: 0.8336580 (187)	total: 2.49s	remaining: 22.3s
400:	learn: 0.8616067	test: 0.8475454	best: 0.8475454 (400)	total: 4.81s	remaining: 19.2s
600:	learn: 0.8746037	test: 0.8625339	best: 0.8633581 (580)	total: 6.74s	remaining: 15.7s
800:	learn: 0.8836525	test: 0.8740766	best: 0.8744453 (798)	total: 8.85s	remaining: 13.2s
1000:	learn: 0.8905771	test: 0.8892145	best: 0.8892145 (960)	total: 10.9s	remaining: 10.9s
1200:	learn: 0.8990099	test: 0.8976220	best: 0.8976220 (1193)	total: 12.9s	remaining: 8.61s
1400:	learn: 0.9051534	test: 0.9013638	best: 0.9022854 (1368)	total: 15s	remaining: 6.41s
1600:	learn: 0.9116909	test: 0.9071881	best: 0.9076495 (1583)	total: 17.8s	remaining: 4.43s
1800:	learn: 0.9172658	test: 0.9126606	best: 0.9126606 (1795)	total: 21.7s	remaining: 2.39s
1999:	learn: 0.9228006	test: 0.9186865	best: 0.9186865 (1993)	total: 24.6s	remaining: 0us

bestT

,class,precision,recall,f1,support
0,AD,0.721429,0.669983,0.694755,603
1,NoDementia_Hy,0.905282,0.935498,0.920142,2217
2,PD_LBD,0.644444,0.500000,0.563107,58
3,Vascular,0.333333,0.133333,0.190476,30


2) Random Forest

This is a good comparison tree baseline.

In [21]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=800,
    max_depth=None,
    min_samples_split=8,
    min_samples_leaf=3,
    max_features="sqrt",
    class_weight={
        "AD": 1.0,
        "NoDementia_Hy": 1.0,
        "PD_LBD": 2.0,
        "Vascular": 4.0,
    },
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

val_proba_rf = rf_model.predict_proba(X_val)
best_vasc_thr_rf = search_best_single_threshold(
    y_val,
    val_proba_rf,
    target_class="Vascular"
)
print("Best RF vascular threshold on VAL:", best_vasc_thr_rf)

test_proba_rf = rf_model.predict_proba(X_test)
test_pred_rf = predict_with_class_thresholds(
    test_proba_rf,
    thresholds={"Vascular": best_vasc_thr_rf["threshold"]}
)

rf_results = eval_predictions(
    y_test,
    test_pred_rf,
    title="Random Forest + tuned vascular threshold"
)
rf_results

Best RF vascular threshold on VAL: {'threshold': 0.24999999999999994, 'precision': 0.9259259259259259, 'recall': 1.0, 'f1': 0.9615384615384616, 'support': 25}

=== Random Forest + tuned vascular threshold ===
               precision    recall  f1-score   support

           AD     0.7266    0.6567    0.6899       603
NoDementia_Hy     0.9026    0.9409    0.9214      2217
       PD_LBD     0.7105    0.4655    0.5625        58
     Vascular     0.3571    0.1667    0.2273        30

     accuracy                         0.8645      2908
    macro avg     0.6742    0.5575    0.6003      2908
 weighted avg     0.8567    0.8645    0.8591      2908

Balanced accuracy: 0.5574528667845505
Macro F1: 0.6002615723821985
Weighted F1: 0.8590596217817317
Confusion matrix:
[[ 396  195    6    6]
 [ 123 2086    5    3]
 [  15   16   27    0]
 [  11   14    0    5]]


,class,precision,recall,f1,support
0,AD,0.726606,0.656716,0.689895,603
1,NoDementia_Hy,0.902640,0.940911,0.921378,2217
2,PD_LBD,0.710526,0.465517,0.562500,58
3,Vascular,0.357143,0.166667,0.227273,30


3) SMOTE for vascular

Use this only on the training set. Do not SMOTE before the split.

In [23]:
from sklearn.impute import SimpleImputer
from collections import Counter

print("Before SMOTE:", Counter(y_train))

imputer = SimpleImputer(strategy="median")

X_train_imp = imputer.fit_transform(X_train)
X_val_imp   = imputer.transform(X_val)
X_test_imp  = imputer.transform(X_test)

print("Any NaN left in train?", np.isnan(X_train_imp).any())
print("Any NaN left in val?", np.isnan(X_val_imp).any())
print("Any NaN left in test?", np.isnan(X_test_imp).any())

Before SMOTE: Counter({'NoDementia_Hy': 8866, 'AD': 2410, 'PD_LBD': 231, 'Vascular': 122})
Any NaN left in train? False
Any NaN left in val? False
Any NaN left in test? False


In [25]:
import numpy as np
from collections import Counter
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE

print("Before SMOTE:", Counter(y_train))

# 1) Impute first
imputer = SimpleImputer(strategy="median")

X_train_imp = imputer.fit_transform(X_train)
X_val_imp   = imputer.transform(X_val)
X_test_imp  = imputer.transform(X_test)

print("NaN in X_train before impute:", np.isnan(np.asarray(X_train, dtype=float)).sum() if hasattr(X_train, "astype") else "check skipped")
print("Any NaN left in X_train_imp?", np.isnan(X_train_imp).any())
print("Any NaN left in X_val_imp?", np.isnan(X_val_imp).any())
print("Any NaN left in X_test_imp?", np.isnan(X_test_imp).any())

# 2) SMOTE on the imputed training data only
smote = SMOTE(
    sampling_strategy={
        "PD_LBD": max(Counter(y_train)["PD_LBD"], 150),
        "Vascular": max(Counter(y_train)["Vascular"], 200),
    },
    random_state=42,
    k_neighbors=3
)

X_train_sm, y_train_sm = smote.fit_resample(X_train_imp, y_train)

print("After SMOTE:", Counter(y_train_sm))
print("X_train_sm shape:", X_train_sm.shape)

Before SMOTE: Counter({'NoDementia_Hy': 8866, 'AD': 2410, 'PD_LBD': 231, 'Vascular': 122})
NaN in X_train before impute: 218470
Any NaN left in X_train_imp? False
Any NaN left in X_val_imp? False
Any NaN left in X_test_imp? False
After SMOTE: Counter({'NoDementia_Hy': 8866, 'AD': 2410, 'PD_LBD': 231, 'Vascular': 200})
X_train_sm shape: (11707, 62)


In [26]:
cat_model_sm = CatBoostClassifier(
    loss_function="MultiClass",
    eval_metric="TotalF1",
    iterations=2000,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=5.0,
    random_seed=42,
    verbose=200,
    class_weights=[1.0, 1.0, 1.5, 2.0]
)

cat_model_sm.fit(
    X_train_sm,
    y_train_sm,
    eval_set=(X_val, y_val),
    use_best_model=True
)

val_proba_cat_sm = cat_model_sm.predict_proba(X_val)
best_vasc_thr_cat_sm = search_best_single_threshold(y_val, val_proba_cat_sm, target_class="Vascular")
print("Best CatBoost+SMOTE vascular threshold on VAL:", best_vasc_thr_cat_sm)

test_proba_cat_sm = cat_model_sm.predict_proba(X_test)
test_pred_cat_sm = predict_with_class_thresholds(
    test_proba_cat_sm,
    thresholds={"Vascular": best_vasc_thr_cat_sm["threshold"]}
)

cat_sm_results = eval_predictions(y_test, test_pred_cat_sm, title="CatBoost + SMOTE + tuned vascular threshold")
cat_sm_results

0:	learn: 0.7939013	test: 0.8052864	best: 0.8052864 (0)	total: 20.7ms	remaining: 41.3s
200:	learn: 0.8511334	test: 0.8504758	best: 0.8504758 (200)	total: 1.52s	remaining: 13.6s
400:	learn: 0.8676579	test: 0.8457393	best: 0.8530298 (260)	total: 3.22s	remaining: 12.8s
600:	learn: 0.8774043	test: 0.8425254	best: 0.8530298 (260)	total: 4.83s	remaining: 11.2s
800:	learn: 0.8842015	test: 0.8426978	best: 0.8530298 (260)	total: 6.37s	remaining: 9.53s
1000:	learn: 0.8893922	test: 0.8417980	best: 0.8530298 (260)	total: 7.83s	remaining: 7.82s
1200:	learn: 0.8965798	test: 0.8410281	best: 0.8530298 (260)	total: 9.31s	remaining: 6.2s
1400:	learn: 0.9030063	test: 0.8424499	best: 0.8530298 (260)	total: 10.8s	remaining: 4.6s
1600:	learn: 0.9076439	test: 0.8419787	best: 0.8530298 (260)	total: 12.3s	remaining: 3.05s
1800:	learn: 0.9132434	test: 0.8451125	best: 0.8530298 (260)	total: 13.7s	remaining: 1.51s
1999:	learn: 0.9169247	test: 0.8487802	best: 0.8530298 (260)	total: 15.2s	remaining: 0us

bestTest =

,class,precision,recall,f1,support
0,AD,0.618280,0.762852,0.682999,603
1,NoDementia_Hy,0.930608,0.883175,0.906272,2217
2,PD_LBD,0.694444,0.431034,0.531915,58
3,Vascular,0.291667,0.233333,0.259259,30


4) One-vs-Rest classifiers

In [47]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

ovr_base = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", OneVsRestClassifier(
        LogisticRegression(
            max_iter=5000,
            solver="saga",
            penalty="l2",
            C=0.1,
            class_weight="balanced"
        )
    ))
])

ovr_base.fit(X_train, y_train)

val_proba_ovr = ovr_base.predict_proba(X_val)
best_vasc_thr_ovr = search_best_single_threshold(y_val, val_proba_ovr, target_class="Vascular")
print("Best OVR-LR vascular threshold on VAL:", best_vasc_thr_ovr)

test_proba_ovr = ovr_base.predict_proba(X_test)
test_pred_ovr = predict_with_class_thresholds(
    test_proba_ovr,
    thresholds={"Vascular": best_vasc_thr_ovr["threshold"]}
)

ovr_results = eval_predictions(y_test, test_pred_ovr, title="One-vs-Rest Logistic + tuned vascular threshold")
ovr_results

c:\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


ValueError: Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

5) Hierarchical model

This is often stronger than flat 4-class classification.

Stage 1: dementia vs no dementia
Stage 2: subtype among dementia only

Use your same class labels. Here I assume NoDementia_Hy is the only non-dementia class.

In [49]:
from catboost import CatBoostClassifier

NO_DEM = "NoDementia_Hy"

# Stage 1 labels: dementia (1) vs no dementia (0)
y_train_s1 = (y_train != NO_DEM).astype(int)
y_val_s1   = (y_val != NO_DEM).astype(int)
y_test_s1  = (y_test != NO_DEM).astype(int)

print("Stage1 class distribution:")
print("train:", np.unique(y_train_s1, return_counts=True))
print("val:", np.unique(y_val_s1, return_counts=True))

stage1 = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="F1",
    iterations=1500,
    learning_rate=0.03,
    depth=6,
    random_seed=42,
    verbose=200,
    class_weights=[1.0, 1.5]
)

stage1.fit(
    X_train,
    y_train_s1,
    eval_set=(X_val, y_val_s1),
    use_best_model=True
)

Stage1 class distribution:
train: (array([0, 1]), array([8866, 2763]))
val: (array([0, 1]), array([1773,  553]))
0:	learn: 0.7650304	test: 0.7651822	best: 0.7651822 (0)	total: 84.5ms	remaining: 2m 6s
200:	learn: 0.8107417	test: 0.7977422	best: 0.7989997 (181)	total: 4.19s	remaining: 27.1s
400:	learn: 0.8357825	test: 0.8266751	best: 0.8266751 (400)	total: 10.5s	remaining: 28.8s
600:	learn: 0.8582474	test: 0.8461538	best: 0.8461538 (600)	total: 19.2s	remaining: 28.7s
800:	learn: 0.8772980	test: 0.8675373	best: 0.8701863 (793)	total: 24.3s	remaining: 21.2s
1000:	learn: 0.8929552	test: 0.8898542	best: 0.8908865 (997)	total: 29.6s	remaining: 14.8s
1200:	learn: 0.9056327	test: 0.9053089	best: 0.9053089 (1199)	total: 36.2s	remaining: 9.01s
1400:	learn: 0.9164022	test: 0.9161012	best: 0.9161012 (1394)	total: 42.1s	remaining: 2.97s
1499:	learn: 0.9194701	test: 0.9199382	best: 0.9209389 (1491)	total: 43.3s	remaining: 0us

bestTest = 0.9209388511
bestIteration = 1491

Shrink model to first 1492 i

CatBoostClassifier(class_weights=[1.0, 1.5], depth=6, eval_metric='F1', iterations=1500, learning_rate=0.03, loss_function='Logloss', random_seed=42, verbose=200)

In [50]:
dementia_classes = ["AD", "PD_LBD", "Vascular"]

DEM_CLASS_TO_LOCAL = {
    "AD":0,
    "PD_LBD":1,
    "Vascular":2
}

LOCAL_TO_GLOBAL = {
    0:"AD",
    1:"PD_LBD",
    2:"Vascular"
}

train_mask_s2 = y_train != NO_DEM
val_mask_s2   = y_val != NO_DEM

X_train_s2 = X_train.loc[train_mask_s2]
X_val_s2   = X_val.loc[val_mask_s2]

y_train_s2 = np.array([DEM_CLASS_TO_LOCAL[v] for v in y_train[train_mask_s2]])
y_val_s2   = np.array([DEM_CLASS_TO_LOCAL[v] for v in y_val[val_mask_s2]])

stage2 = CatBoostClassifier(
    loss_function="MultiClass",
    eval_metric="TotalF1",
    iterations=1500,
    learning_rate=0.03,
    depth=6,
    random_seed=42,
    verbose=200,
    class_weights=[1.0,2.0,4.0]
)

stage2.fit(
    X_train_s2,
    y_train_s2,
    eval_set=(X_val_s2, y_val_s2),
    use_best_model=True
)

0:	learn: 0.7523502	test: 0.7714365	best: 0.7714365 (0)	total: 16.5ms	remaining: 24.8s
200:	learn: 0.8290344	test: 0.8298855	best: 0.8311844 (165)	total: 1.32s	remaining: 8.53s
400:	learn: 0.8835379	test: 0.8567975	best: 0.8648332 (386)	total: 3.87s	remaining: 10.6s
600:	learn: 0.9221401	test: 0.9370410	best: 0.9370410 (595)	total: 6.09s	remaining: 9.11s
800:	learn: 0.9494347	test: 0.9525754	best: 0.9525754 (727)	total: 8.98s	remaining: 7.83s
1000:	learn: 0.9641061	test: 0.9650149	best: 0.9664980 (983)	total: 11.6s	remaining: 5.8s
1200:	learn: 0.9756810	test: 0.9755790	best: 0.9755790 (1161)	total: 14.5s	remaining: 3.6s
1400:	learn: 0.9876248	test: 0.9880232	best: 0.9880232 (1312)	total: 16.4s	remaining: 1.16s
1499:	learn: 0.9897616	test: 0.9894960	best: 0.9894960 (1496)	total: 16.8s	remaining: 0us

bestTest = 0.9894959859
bestIteration = 1496

Shrink model to first 1497 iterations.


CatBoostClassifier(class_weights=[1.0, 2.0, 4.0], depth=6, eval_metric='TotalF1', iterations=1500, learning_rate=0.03, loss_function='MultiClass', random_seed=42, verbose=200)

Tune stage thresholds on validation

In [ ]:
# Stage 1 validation threshold for dementia
val_proba_s1 = stage1.predict_proba(X_val)[:, 1]

best_s1 = None
for thr in np.linspace(0.20, 0.80, 61):
    pred = (val_proba_s1 >= thr).astype(int)
    f1 = f1_score(y_val_s1, pred)
    if best_s1 is None or f1 > best_s1["f1"]:
        best_s1 = {"threshold": thr, "f1": f1}
print("Best Stage1 dementia threshold:", best_s1)

# Stage 2 validation threshold for vascular among dementia cases
val_proba_s2 = stage2.predict_proba(X_val_s2)

best_s2_vasc = None
vasc_local = DEM_CLASS_TO_LOCAL["Vascular"]
for thr in np.linspace(0.05, 0.70, 66):
    pred_local = np.argmax(val_proba_s2, axis=1)
    pred_local[val_proba_s2[:, vasc_local] >= thr] = vasc_local

    p, r, f, s = precision_recall_fscore_support(
        y_val_s2, pred_local, labels=[vasc_local], zero_division=0
    )
    row = {"threshold": thr, "precision": p[0], "recall": r[0], "f1": f[0]}
    if best_s2_vasc is None or row["f1"] > best_s2_vasc["f1"]:
        best_s2_vasc = row
print("Best Stage2 vascular threshold:", best_s2_vasc)

Test-time hierarchical prediction

In [ ]:
def hierarchical_predict(X):
    s1_proba = stage1.predict_proba(X)[:, 1]
    s1_pred = (s1_proba >= best_s1["threshold"]).astype(int)

    final_pred = np.full(len(X), NO_DEM, dtype=int)

    dementia_idx = np.where(s1_pred == 1)[0]
    if len(dementia_idx) > 0:
        X_dem = X.iloc[dementia_idx] if hasattr(X, "iloc") else X[dementia_idx]
        s2_proba = stage2.predict_proba(X_dem)

        s2_pred_local = np.argmax(s2_proba, axis=1)
        s2_pred_local[s2_proba[:, vasc_local] >= best_s2_vasc["threshold"]] = vasc_local

        s2_pred_global = np.array([LOCAL_TO_GLOBAL[v] for v in s2_pred_local])
        final_pred[dementia_idx] = s2_pred_global

    return final_pred

test_pred_hier = hierarchical_predict(X_test)
hier_results = eval_predictions(y_test, test_pred_hier, title="Hierarchical CatBoost")
hier_results

6) Simple vascular feature engineering

7) The “powerful trick”:
Risk-stratified ensemble without changing the model

Step A: define vascular-risk strata

In [28]:
def compute_vascular_risk_stratum(df):
    df = add_vascular_features(df)

    if "vascular_burden_score" not in df.columns:
        raise ValueError("vascular_burden_score was not created; check available columns.")

    # Example bins:
    # 0-1 -> low
    # 2-3 -> medium
    # 4+  -> high
    score = df["vascular_burden_score"].fillna(0)

    stratum = pd.Series(index=df.index, dtype="object")
    stratum[score <= 1] = "low"
    stratum[(score >= 2) & (score <= 3)] = "medium"
    stratum[score >= 4] = "high"
    return stratum

Step B: learn best vascular threshold inside each stratum using validation predictions

In [ ]:
def fit_stratum_specific_thresholds(y_val, val_proba, val_df, target_class="Vascular"):
    j = CLASS_TO_INT[target_class]
    strata = compute_vascular_risk_stratum(val_df)

    thresholds = {}
    summary_rows = []

    for s in ["low", "medium", "high"]:
        mask = (strata == s).values
        y_sub = y_val[mask]
        proba_sub = val_proba[mask]

        if len(y_sub) == 0:
            continue

        best = None
        for thr in np.linspace(0.05, 0.70, 66):
            pred = np.argmax(proba_sub, axis=1)
            pred[proba_sub[:, j] >= thr] = j

            p, r, f, sup = precision_recall_fscore_support(
                y_sub, pred, labels=[j], zero_division=0
            )
            row = {
                "stratum": s,
                "threshold": thr,
                "precision": p[0],
                "recall": r[0],
                "f1": f[0],
                "support": sup[0]
            }
            if best is None or row["f1"] > best["f1"]:
                best = row

        thresholds[s] = best["threshold"]
        summary_rows.append(best)

    return thresholds, pd.DataFrame(summary_rows)

# Example using CatBoost validation probabilities:
val_proba_cat = cat_model.predict_proba(X_val_fe)
stratum_thresholds, stratum_threshold_table = fit_stratum_specific_thresholds(
    y_val, val_proba_cat, X_val_fe, target_class="Vascular"
)

print("Stratum-specific vascular thresholds:", stratum_thresholds)
display(stratum_threshold_table)

NameError: name 'X_val_fe' is not defined

Step C: apply the stratified rule on test set

In [ ]:
def predict_with_risk_stratified_vascular_rule(proba, X_df, stratum_thresholds):
    pred = np.argmax(proba, axis=1)
    vasc_idx = CLASS_TO_INT["Vascular"]

    strata = compute_vascular_risk_stratum(X_df)

    for s in ["low", "medium", "high"]:
        mask = (strata == s).values
        if s in stratum_thresholds:
            thr = stratum_thresholds[s]
            pred[np.where(mask)[0][proba[mask, vasc_idx] >= thr]] = vasc_idx

    return pred

test_proba_cat = cat_model.predict_proba(X_test_fe)
test_pred_strat = predict_with_risk_stratified_vascular_rule(
    test_proba_cat,
    X_test_fe,
    stratum_thresholds
)

strat_results = eval_predictions(
    y_test,
    test_pred_strat,
    title="CatBoost + risk-stratified vascular thresholding"
)
strat_results